In [1]:
import glob
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error


In [2]:
import os
import re
import pandas as pd

num_words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
             "thirteen": 13, "fourteen": 14, "fifteen": 15}

pat = re.compile(
    r"^\s*(\d{1,2})\.\s+(.+?),\s*([A-Z0-9/]+):\s*(\d+|[A-Za-z]+)[\s-]+years?,\s*\$([\d.]+)\s*MM",
    re.IGNORECASE)

cols = ["player_name", "signing_year", "mlbtr_rank", "mlbtr_pos", "mlbtr_years", "mlbtr_total_m"]

def parse_mlbtr(path, signing_year):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = pat.match(line)
            if not m:
                continue
            rank, name, pos, yrs, total = m.groups()
            yrs = int(yrs) if yrs.isdigit() else num_words.get(yrs.lower())
            if yrs is None:
                print("Could not read years in:", line.strip()[:80])
                continue
            rows.append([name.strip(), signing_year, int(rank), pos.upper(), yrs, float(total)])
    out = pd.DataFrame(rows, columns=cols)
    out["mlbtr_aav_m"] = (out["mlbtr_total_m"] / out["mlbtr_years"]).round(2)
    return out

frames = []
for yr in [2024, 2025, 2026]:
    path = f"../data/raw/mlbtr_pred_{yr}.txt"
    if not os.path.exists(path):
        print(yr, "- file not found yet, skipping")
        continue
    df_yr = parse_mlbtr(path, yr)
    print(yr, "-", len(df_yr), "predictions parsed")
    frames.append(df_yr)

if frames:
    mlbtr = pd.concat(frames, ignore_index=True)
    mlbtr.to_csv("../data/processed/mlbtr_predictions.csv", index=False)
    display(mlbtr[mlbtr["signing_year"] == 2026].head(6))

2024 - 0 predictions parsed
2025 - 0 predictions parsed
2026 - 50 predictions parsed


,player_name,signing_year,mlbtr_rank,mlbtr_pos,mlbtr_years,mlbtr_total_m,mlbtr_aav_m
0,Kyle Tucker,2026,1,RF,11,400.0,36.36
1,Bo Bichette,2026,2,SS,8,208.0,26.0
2,Dylan Cease,2026,3,SP,7,189.0,27.0
3,Munetaka Murakami,2026,4,3B/1B,8,180.0,22.5
4,Alex Bregman,2026,5,3B,6,160.0,26.67
5,Framber Valdez,2026,6,SP,5,150.0,30.0


In [3]:
for yr in [2024, 2025]:
    with open(f"../data/raw/mlbtr_pred_{yr}.txt", encoding="utf-8") as f:
        text = f.read()
    lines = [l for l in text.splitlines() if l.strip()]
    print(f"{yr}: {len(text)} characters, {len(lines)} non-empty lines")
    for l in lines[:6]:
        print("   ", repr(l[:100]))
    print()

2024: 125816 characters, 426 non-empty lines
    '1. Shohei Ohtani. Twelve years, $528MM'
    'Tim Dierkes’ prediction: Dodgers / Anthony Franco’s prediction: Dodgers / Darragh McDonald’s predict'
    'Ohtani’s two-way legend stretches back to his days in Japan. He’d hit .286/.358/.501 while pitching '
    'Ohtani was arguably the most anticipated international signee ever. There was never any question the'
    'He dispelled some concern with a strong rookie showing. Ohtani connected on 22 homers in 114 games t'
    'Ohtani had a fine second year as a hitter. His return to the mound in the shortened 2020 campaign wa'

2025: 128137 characters, 372 non-empty lines
    '1. Juan Soto: Thirteen years, $600MM'
    'Tim Dierkes’ prediction: Yankees / Anthony Franco’s prediction: Yankees / Darragh McDonald’s predict'
    'Soto isn’t just one of the best hitters on the planet, he’s one of the youngest and thus most appeal'
    'It’s rare for a player to even make his big league debut at 19, let

In [4]:
import os
import re
import pandas as pd

num_words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
             "thirteen": 13, "fourteen": 14, "fifteen": 15}

# rank. Name[, POS] then ":" or "." then "N years, $XMM" (years may be a word)
pat = re.compile(
    r"^\s*(\d{1,2})\.\s+(.+?)(?:,\s*([A-Z0-9/]+))?\s*[.:]\s*(\d+|[A-Za-z]+)[\s-]+years?,\s*\$([\d.,]+)\s*MM",
    re.IGNORECASE)

cols = ["player_name", "signing_year", "mlbtr_rank", "mlbtr_pos", "mlbtr_years", "mlbtr_total_m"]

def parse_mlbtr(path, signing_year):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = pat.match(line)
            if not m:
                continue
            rank, name, pos, yrs, total = m.groups()
            yrs = int(yrs) if yrs.isdigit() else num_words.get(yrs.lower())
            if yrs is None:
                print("Could not read years in:", line.strip()[:80])
                continue
            rows.append([name.strip(), signing_year, int(rank), (pos or "").upper(),
                         yrs, float(total.replace(",", ""))])
    out = pd.DataFrame(rows, columns=cols)
    out["mlbtr_aav_m"] = (out["mlbtr_total_m"] / out["mlbtr_years"]).round(2)
    return out

frames = []
for yr in [2024, 2025, 2026]:
    path = f"../data/raw/mlbtr_pred_{yr}.txt"
    if not os.path.exists(path):
        print(yr, "- file not found yet, skipping")
        continue
    df_yr = parse_mlbtr(path, yr)
    print(yr, "-", len(df_yr), "predictions parsed", "(expected 50)" if len(df_yr) != 50 else "")
    frames.append(df_yr)

if frames:
    mlbtr = pd.concat(frames, ignore_index=True)
    mlbtr.to_csv("../data/processed/mlbtr_predictions.csv", index=False)
    display(mlbtr.groupby("signing_year").head(3))

2024 - 50 predictions parsed 
2025 - 50 predictions parsed 
2026 - 50 predictions parsed 


,player_name,signing_year,mlbtr_rank,mlbtr_pos,mlbtr_years,mlbtr_total_m,mlbtr_aav_m
0,Shohei Ohtani,2024,1,,12,528.0,44.00
1,Cody Bellinger,2024,2,,12,264.0,22.00
2,Yoshinobu Yamamoto,2024,3,,9,225.0,25.00
50,Juan Soto,2025,1,,13,600.0,46.15
51,Corbin Burnes,2025,2,,7,200.0,28.57
52,Alex Bregman,2025,3,,7,182.0,26.00
100,Kyle Tucker,2026,1,RF,11,400.0,36.36
101,Bo Bichette,2026,2,SS,8,208.0,26.00
102,Dylan Cease,2026,3,SP,7,189.0,27.00


In [5]:
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015
d["key"] = d["player_name"].apply(norm)

def savant_avg(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, sy in zip(df["mlb_ID"], df["stat_year"]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(sy) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def new_model():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))

def oos_predictions(t):
    f = feats[t]
    df = d[d["player_type"] == t].copy()
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    has_sav = df[f[0]].notna().values
    df[f] = df[f].fillna(50)
    colsA = ["age_at_signing", "age_sq", "qualifying_offer", "year", "war_w"]
    colsB = colsA + f
    XA = pd.get_dummies(df[colsA + ["position"]], columns=["position"], dtype=float)
    XB = pd.get_dummies(df[colsB + ["position"]], columns=["position"], dtype=float)
    out = []
    for yr in [2024, 2025, 2026]:
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        mA = new_model().fit(XA[tr], df.loc[tr, "aav_m"])
        mB = new_model().fit(XB[tr & has_sav], df.loc[tr & has_sav, "aav_m"])
        part = df[te].copy()
        part["model_aav_m"] = np.where(has_sav[te], mB.predict(XB[te]), mA.predict(XA[te]))
        out.append(part)
    return pd.concat(out)

oos = pd.concat([oos_predictions("hitter"), oos_predictions("pitcher")])
oos = (oos.rename(columns={"aav_m": "actual_aav_m", "years": "actual_years",
                           "total_value": "actual_total"})
          [["key", "signing_year", "actual_aav_m", "actual_years", "actual_total", "model_aav_m"]]
          .drop_duplicates(["key", "signing_year"]))
oos["actual_total_m"] = oos["actual_total"] / 1e6

mt = pd.read_csv("../data/processed/mlbtr_predictions.csv")
mt["key"] = mt["player_name"].apply(norm)
cmp = mt.merge(oos, on=["key", "signing_year"], how="left")

print("MLBTR top 50 -> players with an actual contract the model can score:")
for yr, g in cmp.groupby("signing_year"):
    print(f"  {yr}: {g['actual_aav_m'].notna().sum()} of {len(g)}")
    print("     not scored:", ", ".join(g.loc[g["actual_aav_m"].isna(), "player_name"]))

m = cmp[cmp["actual_aav_m"].notna()].copy()
m["mlbtr_err"] = m["mlbtr_aav_m"] - m["actual_aav_m"]
m["model_err"] = m["model_aav_m"] - m["actual_aav_m"]

def summarize(g):
    return pd.Series({
        "n": len(g),
        "MLBTR MAE": g["mlbtr_err"].abs().mean(),
        "Model MAE": g["model_err"].abs().mean(),
        "MLBTR bias": g["mlbtr_err"].mean(),
        "Model bias": g["model_err"].mean(),
    })

print("\nAAV error in $M (bias = predicted minus actual, negative = too low):")
print(m.groupby("signing_year").apply(summarize).round(2))
print("\nAll three classes pooled:")
print(summarize(m).round(2).to_string())

print("\nMLBTR on contract structure (for context):")
print("  average miss on years: %.1f" % (m["mlbtr_years"] - m["actual_years"]).abs().mean())
print("  average miss on total value: $%.0fM" % (m["mlbtr_total_m"] - m["actual_total_m"]).abs().mean())

m.to_csv("../outputs/benchmark_mlbtr_vs_model.csv", index=False)
m.sort_values("actual_aav_m", ascending=False)[
    ["player_name", "signing_year", "actual_aav_m", "mlbtr_aav_m", "model_aav_m"]].head(15).round(1)

MLBTR top 50 -> players with an actual contract the model can score:
  2024: 37 of 50
     not scored: Shohei Ohtani, Yoshinobu Yamamoto, Shota Imanaga, Jung Hoo Lee, Yariel Rodriguez, Mike Clevinger, Michael Lorenzen, Gio Urshela, Amed Rosario, Jason Heyward, Yuki Matsui, Brandon Belt, Jakob Junis
  2025: 44 of 50
     not scored: Hyeseong Kim, Nick Martinez, Jose Quintana, Tomoyuki Sugano, Shinnosuke Ogasawara, Spencer Turnbull
  2026: 39 of 50
     not scored: Munetaka Murakami, Tatsuya Imai, Zac Gallen, Brandon Woodruff, Trent Grisham, Kazuma Okamoto, Shota Imanaga, Lucas Giolito, Gleyber Torres, Cody Ponce, Max Scherzer

AAV error in $M (bias = predicted minus actual, negative = too low):
                 n  MLBTR MAE  Model MAE  MLBTR bias  Model bias
signing_year                                                    
2024          37.0       2.19       3.34        0.55       -1.71
2025          44.0       3.44       4.21       -1.56       -2.22
2026          39.0       3.72       4

,player_name,signing_year,actual_aav_m,mlbtr_aav_m,model_aav_m
100,Kyle Tucker,2026,60.0,36.4,33.1
50,Juan Soto,2025,51.0,46.2,39.0
101,Bo Bichette,2026,42.0,26.0,26.3
52,Alex Bregman,2025,40.0,26.0,24.0
105,Framber Valdez,2026,38.3,30.0,26.4
53,Blake Snell,2025,36.4,32.0,22.3
104,Alex Bregman,2026,35.0,26.7,20.0
51,Corbin Burnes,2025,35.0,28.6,28.1
107,Cody Bellinger,2026,32.5,28.0,20.7
3,Blake Snell,2024,31.0,28.6,26.8


In [6]:
import re, unicodedata
import pandas as pd

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

# 1. Why are these domestic players unscored?
c = pd.read_csv("../data/processed/contracts.csv")
c["key"] = c["player_name"].apply(norm)
check = {
    2024: ["Mike Clevinger", "Michael Lorenzen", "Gio Urshela", "Amed Rosario",
           "Jason Heyward", "Brandon Belt", "Jakob Junis"],
    2025: ["Jose Quintana", "Spencer Turnbull"],
    2026: ["Lucas Giolito", "Max Scherzer"],
}
rows = []
for yr, names in check.items():
    for n in names:
        hit = c[(c["key"] == norm(n)) & (c["signing_year"] == yr)]
        rows.append({"player": n, "class": yr, "in_contracts_csv": len(hit) > 0,
                     "total_m": round(hit["total_value"].iloc[0] / 1e6, 1) if len(hit) else None,
                     "years": hit["years"].iloc[0] if len(hit) else None})
print(pd.DataFrame(rows).to_string(index=False))

# 2. How often was MLBTR closer, and does it depend on contract size?
m = pd.read_csv("../outputs/benchmark_mlbtr_vs_model.csv")
closer = m["mlbtr_err"].abs() < m["model_err"].abs()
print("\nMLBTR closer on %d of %d players (%.0f%%)" % (closer.sum(), len(m), 100 * closer.mean()))
m["size"] = pd.cut(m["actual_aav_m"], [0, 10, 20, 100], labels=["under $10M", "$10-20M", "$20M+"])
print(m.groupby("size", observed=True).agg(
    n=("mlbtr_err", "size"),
    mlbtr_mae=("mlbtr_err", lambda s: s.abs().mean()),
    model_mae=("model_err", lambda s: s.abs().mean()),
    mlbtr_bias=("mlbtr_err", "mean"),
    model_bias=("model_err", "mean")).round(2))

          player  class  in_contracts_csv  total_m  years
  Mike Clevinger   2024              True      3.0    1.0
Michael Lorenzen   2024              True      4.5    1.0
     Gio Urshela   2024             False      NaN    NaN
    Amed Rosario   2024              True      1.5    1.0
   Jason Heyward   2024             False      NaN    NaN
    Brandon Belt   2024             False      NaN    NaN
     Jakob Junis   2024             False      NaN    NaN
   Jose Quintana   2025              True      4.0    1.0
Spencer Turnbull   2025              True      1.3    1.0
   Lucas Giolito   2026              True      3.0    1.0
    Max Scherzer   2026              True      3.0    1.0

MLBTR closer on 73 of 120 players (61%)
             n  mlbtr_mae  model_mae  mlbtr_bias  model_bias
size                                                        
under $10M  21       2.36       2.60        1.34        2.20
$10-20M     69       2.58       2.84       -0.15       -1.43
$20M+       30     

In [7]:
import glob, re
import pandas as pd

rows = []
for path in sorted(glob.glob("../data/processed/contracts_20??.csv")):
    yr = int(re.search(r"contracts_(\d{4})\.csv$", path.replace("\\", "/")).group(1))
    raw = pd.read_csv(path).dropna(how="all")
    pcol = [c for c in raw.columns if c.startswith("Player")][0]
    vcol = [c for c in raw.columns if c.startswith("Value")][0]
    val = pd.to_numeric(raw[vcol].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
    yrs = pd.to_numeric(raw["Yrs"], errors="coerce")
    zero = raw[(val.fillna(0) <= 0) | (yrs.fillna(0) <= 0)]
    names = ", ".join(zero[pcol].astype(str).str.replace(" QO", "", regex=False))
    rows.append((yr, len(raw), len(zero), names))

audit = pd.DataFrame(rows, columns=["class", "rows", "zero_value_rows", "names"])
print(audit[["class", "rows", "zero_value_rows"]].to_string(index=False))
for _, r in audit.iterrows():
    print(f"\n{r['class']}: {r['names']}")

 class  rows  zero_value_rows
  2015   108                0
  2016   144                0
  2017   103                0
  2018   102                0
  2019   112                0
  2020   133                0
  2021   159                0
  2022   155                0
  2023   154                5
  2024   168               15
  2025   173                0
  2026   160                1

2015: 

2016: 

2017: 

2018: 

2019: 

2020: 

2021: 

2022: 

2023: Emailin Montilla, Josimar Cousin, Jorge Alfaro, Brent Honeywell, Ryan Tepera

2024: Gerardo Gutierrez, Emmanuel Chapman, Luis Gomez, Tomás Nido, Logan Porter, Drew Pomeranz, Andrew Knapp, Brian Anderson, Giovanny Urshela, Whit Merrifield, Jason Heyward, Ali Sanchez, Jason Heyward, Kirby Yates, Jackson Stephens

2025: 

2026: Craig Kimbrel
